# Notebook 127: Transformerアーキテクチャをゼロから実装
## Transformer Architecture from Scratch
---
### このノートブックの位置づけ
**Unit 8「シーケンスモデリング」** の発展編。NB126で学んだAttentionメカニズム（特にScaled Dot-Product AttentionとSelf-Attention）を土台に、**Transformer** アーキテクチャをゼロから構築します。

### NB126からの接続
NB126では以下を学びました：
- **Bahdanau Attention** でSeq2Seqのボトルネック問題を解決
- **Scaled Dot-Product Attention** の数式と実装
- **Self-Attention** の概念とTransformerへの布石

本ノートブックでは、これらの知識を基に **"Attention Is All You Need"** (Vaswani et al., 2017) のTransformerを完全に実装します。

### 学習目標
1. **Self-Attentionの行列計算** を図と数式で完全に理解する
2. **Scaled Dot-Product Attention** をnumpyでステップバイステップ実装できる
3. **Multi-Head Attention** の仕組みを理解し、PyTorchで実装できる
4. **Positional Encoding** の数学的背景を理解し、可視化できる
5. **Transformerブロック** を組み立て、完全なモデルを構築できる
6. 簡単なタスクでTransformerの動作を確認できる

### 前提知識
- Notebook 126（Attention、Scaled Dot-Product Attention、Self-Attention）
- PyTorchの基礎（nn.Module、テンソル操作）

⏱️ **推定学習時間**: 180-240分  
📊 **難易度**: ★★★★★（上級）  
🎓 **カテゴリ**: 発展

## 目次

1. [Self-Attentionの行列計算を図解](#1.-Self-Attentionの行列計算を図解)
2. [Scaled Dot-Product Attentionの実装（NumPy）](#2.-Scaled-Dot-Product-Attentionの実装（NumPy）)
3. [Multi-Head Attentionの実装（PyTorch）](#3.-Multi-Head-Attentionの実装（PyTorch）)
4. [Positional Encodingの実装と可視化](#4.-Positional-Encodingの実装と可視化)
5. [Transformerブロックの実装](#5.-Transformerブロックの実装)
6. [完全なTransformerモデル](#6.-完全なTransformerモデル)
7. [動作確認：文字列コピー・逆順タスク](#7.-動作確認：文字列コピー・逆順タスク)
8. [まとめと次のステップ](#8.-まとめと次のステップ)
9. [自己評価クイズ](#9.-自己評価クイズ)

In [ ]:
# ============================================================
# セットアップ
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import warnings
warnings.filterwarnings('ignore')

import japanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cpu')
print(f"PyTorch version: {torch.__version__}")
print("環境セットアップ完了")

# ============================================================
# 1. Self-Attentionの行列計算を図解
# ============================================================

## 1.1 Attentionの全体像

NB126で学んだScaled Dot-Product Attentionを復習しましょう。Transformerの核となる計算は次の式です：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### 各行列の役割

| 行列 | 名前 | 役割 | 直感的な意味 |
|------|------|------|-------------|
| $Q$ | Query | 「何を探しているか」 | 質問 |
| $K$ | Key | 「何を持っているか」 | 索引 |
| $V$ | Value | 「実際の情報」 | 内容 |

### Self-Attentionの特徴
**Self-Attention** では、Q, K, V がすべて **同じ入力** から生成されます。つまり、各トークンが「他のすべてのトークンとの関係」を計算します。

## 1.2 行列計算のステップバイステップ図解

入力系列を $X \in \mathbb{R}^{n \times d_{\text{model}}}$ とします（$n$: 系列長、$d_{\text{model}}$: 埋め込み次元）。

### Step 1: 線形変換で Q, K, V を生成

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

ここで $W^Q, W^K \in \mathbb{R}^{d_{\text{model}} \times d_k}$、$W^V \in \mathbb{R}^{d_{\text{model}} \times d_v}$

### Step 2: Attention Scoreの計算

$$\text{Score} = QK^T \in \mathbb{R}^{n \times n}$$

この行列の $(i, j)$ 要素は、トークン $i$ がトークン $j$ にどれだけ注目するかを表します。

### Step 3: スケーリングとSoftmax

$$\text{Weights} = \text{softmax}\left(\frac{\text{Score}}{\sqrt{d_k}}\right)$$

$\sqrt{d_k}$ で割る理由：$d_k$ が大きいとき、内積の値も大きくなり、softmaxが極端な分布になってしまう（勾配消失）。

### Step 4: 重み付き和

$$\text{Output} = \text{Weights} \cdot V \in \mathbb{R}^{n \times d_v}$$

In [ ]:
# ============================================================
# 1.3 行列計算の可視化
# ============================================================
# 具体例で行列計算を可視化する

# 設定: 3トークンの系列、埋め込み次元=4
n_tokens = 3
d_model = 4
d_k = d_v = 3  # Q,K,Vの次元

# 入力行列 X (各行が1トークンの埋め込み)
X = np.array([
    [1.0, 0.5, 0.2, 0.1],   # "I"
    [0.3, 1.0, 0.8, 0.4],   # "love"
    [0.7, 0.2, 1.0, 0.6],   # "AI"
])
token_labels = ["I", "love", "AI"]

# 重み行列（簡略化のため小さな値）
np.random.seed(42)
W_Q = np.random.randn(d_model, d_k) * 0.5
W_K = np.random.randn(d_model, d_k) * 0.5
W_V = np.random.randn(d_model, d_v) * 0.5

# Step 1: Q, K, V の計算
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Step 2: Attention Score
scores = Q @ K.T

# Step 3: スケーリング + Softmax
scaled_scores = scores / np.sqrt(d_k)

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

weights = softmax(scaled_scores)

# Step 4: 出力
output = weights @ V

# 可視化
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Q*K^T (スコア)
im0 = axes[0].imshow(scores, cmap='RdBu_r', aspect='auto')
axes[0].set_title('Step 2: QK^T (生スコア)', fontsize=12)
axes[0].set_xticks(range(n_tokens))
axes[0].set_yticks(range(n_tokens))
axes[0].set_xticklabels(token_labels)
axes[0].set_yticklabels(token_labels)
for i in range(n_tokens):
    for j in range(n_tokens):
        axes[0].text(j, i, f'{scores[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Scaled scores
im1 = axes[1].imshow(scaled_scores, cmap='RdBu_r', aspect='auto')
axes[1].set_title(f'Step 3a: Score / sqrt({d_k})', fontsize=12)
axes[1].set_xticks(range(n_tokens))
axes[1].set_yticks(range(n_tokens))
axes[1].set_xticklabels(token_labels)
axes[1].set_yticklabels(token_labels)
for i in range(n_tokens):
    for j in range(n_tokens):
        axes[1].text(j, i, f'{scaled_scores[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Attention weights (softmax後)
im2 = axes[2].imshow(weights, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
axes[2].set_title('Step 3b: Softmax (Attention重み)', fontsize=12)
axes[2].set_xticks(range(n_tokens))
axes[2].set_yticks(range(n_tokens))
axes[2].set_xticklabels(token_labels)
axes[2].set_yticklabels(token_labels)
for i in range(n_tokens):
    for j in range(n_tokens):
        axes[2].text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im2, ax=axes[2], shrink=0.8)

# 出力
im3 = axes[3].imshow(output, cmap='viridis', aspect='auto')
axes[3].set_title('Step 4: Weights @ V (出力)', fontsize=12)
axes[3].set_xticks(range(d_v))
axes[3].set_yticks(range(n_tokens))
axes[3].set_xticklabels([f'd_{i}' for i in range(d_v)])
axes[3].set_yticklabels(token_labels)
for i in range(n_tokens):
    for j in range(d_v):
        axes[3].text(j, i, f'{output[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im3, ax=axes[3], shrink=0.8)

plt.suptitle('Self-Attention 行列計算の流れ', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n各行の重みの合計（=1.0になるはず）:")
for i, label in enumerate(token_labels):
    print(f"  {label}: {weights[i].sum():.4f}")

### 図の読み方

- **QK^T**: 各トークンペア間の「相性スコア」。値が大きいほど関連が強い
- **スケーリング後**: $\sqrt{d_k}$ で割ることで値の範囲を穏やかにする
- **Softmax後**: 各行が確率分布になる（合計=1.0）。これが「Attention重み」
- **出力**: Attention重みでValueを重み付き平均した結果

> **ポイント**: Self-Attentionは「各トークンが他のすべてのトークンの情報を加重平均で集約する」メカニズムです。

# ============================================================
# 2. Scaled Dot-Product Attentionの実装（NumPy）
# ============================================================

NB126ではPyTorchで実装しましたが、ここではnumpyで **ゼロから** 実装し、各ステップの計算を完全に理解します。

In [ ]:
# ============================================================
# 2.1 NumPyによるScaled Dot-Product Attention
# ============================================================

def scaled_dot_product_attention_numpy(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention (NumPy実装)
    
    Parameters:
    -----------
    Q : np.ndarray, shape (..., seq_len_q, d_k)
    K : np.ndarray, shape (..., seq_len_k, d_k)
    V : np.ndarray, shape (..., seq_len_k, d_v)
    mask : np.ndarray or None, shape (..., seq_len_q, seq_len_k)
        マスク（Trueの位置を-infに設定）
    
    Returns:
    --------
    output : np.ndarray, shape (..., seq_len_q, d_v)
    attention_weights : np.ndarray, shape (..., seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    
    # Step 1: QK^T を計算
    scores = Q @ K.swapaxes(-2, -1)  # (..., seq_len_q, seq_len_k)
    
    # Step 2: スケーリング
    scores = scores / np.sqrt(d_k)
    
    # Step 3: マスク適用（オプション）
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    
    # Step 4: Softmax
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    
    # Step 5: 重み付き和
    output = attention_weights @ V  # (..., seq_len_q, d_v)
    
    return output, attention_weights

# テスト: 先ほどと同じ入力で検証
output_np, weights_np = scaled_dot_product_attention_numpy(Q, K, V)

print("=== NumPy実装の検証 ===")
print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"Output shape: {output_np.shape}")
print(f"Weights shape: {weights_np.shape}")
print(f"\n先ほどの結果と一致: {np.allclose(output_np, output)}")
print(f"重みの行和 = 1.0: {np.allclose(weights_np.sum(axis=-1), 1.0)}")

## 2.2 マスク付きAttention（因果マスク）

Transformerの **デコーダ** では、未来のトークンを見ないように **因果マスク（causal mask）** を使います。これは下三角行列のマスクです。

In [ ]:
# ============================================================
# 2.2 因果マスク（Causal Mask）の実装と効果
# ============================================================

seq_len = 5
# 因果マスク: 上三角部分をTrue（=マスクする位置）
causal_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)

print("因果マスク（Trueの位置が遮蔽される）:")
print(causal_mask.astype(int))
print()

# テスト用のQ, K, V
np.random.seed(123)
d_k_test = 4
Q_test = np.random.randn(seq_len, d_k_test)
K_test = np.random.randn(seq_len, d_k_test)
V_test = np.random.randn(seq_len, d_k_test)

# マスクなし vs マスクあり
out_no_mask, w_no_mask = scaled_dot_product_attention_numpy(Q_test, K_test, V_test)
out_masked, w_masked = scaled_dot_product_attention_numpy(Q_test, K_test, V_test, mask=causal_mask)

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(causal_mask.astype(float), cmap='Greys', vmin=0, vmax=1)
axes[0].set_title('因果マスク\n(白=通過, 黒=遮蔽)', fontsize=12)
for i in range(seq_len):
    for j in range(seq_len):
        axes[0].text(j, i, str(causal_mask[i,j].astype(int)), ha='center', va='center',
                    color='white' if causal_mask[i,j] else 'black', fontsize=11)
axes[0].set_xlabel('Key位置')
axes[0].set_ylabel('Query位置')

im1 = axes[1].imshow(w_no_mask, cmap='YlOrRd', vmin=0, vmax=1)
axes[1].set_title('マスクなし Attention重み', fontsize=12)
for i in range(seq_len):
    for j in range(seq_len):
        axes[1].text(j, i, f'{w_no_mask[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(w_masked, cmap='YlOrRd', vmin=0, vmax=1)
axes[2].set_title('因果マスク付き Attention重み', fontsize=12)
for i in range(seq_len):
    for j in range(seq_len):
        axes[2].text(j, i, f'{w_masked[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im2, ax=axes[2], shrink=0.8)

plt.suptitle('因果マスクの効果', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("マスク付き: 各トークンは自分自身と過去のトークンにのみ注目")
print("→ デコーダでの自己回帰生成に必要")

# ============================================================
# 3. Multi-Head Attentionの実装（PyTorch）
# ============================================================

## 3.1 なぜMulti-Headが必要か？

単一のAttentionヘッドでは、1つの「関係性の観点」しか捉えられません。例えば：
- ヘッド1: **構文的な関係**（主語-動詞）
- ヘッド2: **意味的な関係**（同義語、関連語）
- ヘッド3: **位置的な近さ**

Multi-Head Attentionは **複数の「注目の仕方」を並列に計算** し、結合します。

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) W^O$$

$$\text{where } \text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

### パラメータの関係
- $d_{\text{model}}$: モデルの次元（例: 512）
- $h$: ヘッド数（例: 8）
- $d_k = d_v = d_{\text{model}} / h$: 各ヘッドの次元（例: 64）

In [ ]:
# ============================================================
# 3.2 Multi-Head Attention の PyTorch実装
# ============================================================

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention (自作実装)"""
    
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # 各ヘッドの次元
        
        # Q, K, V の線形変換
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        
        # 出力の線形変換
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        
    def forward(self, Q, K, V, mask=None):
        """
        Parameters:
        -----------
        Q, K, V : Tensor, shape (batch, seq_len, d_model)
        mask : Tensor or None, shape (batch, 1, seq_len_q, seq_len_k)
        
        Returns:
        --------
        output : Tensor, shape (batch, seq_len_q, d_model)
        attention_weights : Tensor, shape (batch, n_heads, seq_len_q, seq_len_k)
        """
        batch_size = Q.size(0)
        
        # Step 1: 線形変換
        Q = self.W_Q(Q)  # (batch, seq_len, d_model)
        K = self.W_K(K)
        V = self.W_V(V)
        
        # Step 2: ヘッドに分割
        # (batch, seq_len, d_model) -> (batch, n_heads, seq_len, d_k)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Step 3: Scaled Dot-Product Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V)
        
        # Step 4: ヘッドを結合
        # (batch, n_heads, seq_len, d_k) -> (batch, seq_len, d_model)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # Step 5: 出力線形変換
        output = self.W_O(context)
        
        return output, attention_weights

# テスト
d_model = 32
n_heads = 4
batch_size = 2
seq_len = 5

mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(batch_size, seq_len, d_model)

output, attn_weights = mha(x, x, x)  # Self-Attention

print("=== Multi-Head Attention テスト ===")
print(f"入力: {x.shape}")
print(f"出力: {output.shape}")
print(f"Attention重み: {attn_weights.shape}")
print(f"パラメータ数: {sum(p.numel() for p in mha.parameters()):,}")

In [ ]:
# ============================================================
# 3.3 各ヘッドのAttentionパターンを可視化
# ============================================================

# わかりやすい例: 固定入力で各ヘッドの振る舞いを確認
torch.manual_seed(42)
mha_viz = MultiHeadAttention(d_model=16, n_heads=4)

# 8トークンの入力を作成
seq_len_viz = 8
x_viz = torch.randn(1, seq_len_viz, 16)
_, attn_viz = mha_viz(x_viz, x_viz, x_viz)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for h in range(4):
    w = attn_viz[0, h].detach().numpy()
    im = axes[h].imshow(w, cmap='YlOrRd', vmin=0, vmax=w.max())
    axes[h].set_title(f'Head {h+1}', fontsize=13)
    axes[h].set_xlabel('Key位置')
    if h == 0:
        axes[h].set_ylabel('Query位置')
    plt.colorbar(im, ax=axes[h], shrink=0.8)

plt.suptitle('Multi-Head Attention: 各ヘッドのAttentionパターン\n（初期化直後・学習前）',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("各ヘッドが異なるAttentionパターンを学習する可能性を持つ")
print("学習後は、構文・意味・位置など異なる関係性を捉えるようになる")

# ============================================================
# 4. Positional Encodingの実装と可視化
# ============================================================

## 4.1 なぜ位置情報が必要か？

Self-Attentionは **集合に対する操作** であり、トークンの順序情報を持ちません。
入力の順番を入れ替えても（Query/Key/Valueの行を入れ替えても）、対応する出力も同じように入れ替わるだけです。

しかし自然言語では語順が重要です：
- "犬が猫を追いかけた" ≠ "猫が犬を追いかけた"

**Positional Encoding** は位置情報を埋め込みに加算することで、順序を認識させます。

## 4.2 正弦波Positional Encoding

Vaswani et al. (2017) は以下の正弦波関数を提案しました：

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

### この設計の利点
1. **任意の長さに対応**: 学習なしで任意の位置に対応
2. **相対位置の表現**: $PE_{pos+k}$ は $PE_{pos}$ の線形変換で表せる
3. **有界**: 値が $[-1, 1]$ の範囲に収まる

In [ ]:
# ============================================================
# 4.3 Positional Encoding の実装
# ============================================================

class PositionalEncoding(nn.Module):
    """正弦波 Positional Encoding"""
    
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Positional Encoding行列を事前計算
        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)
        
        # 10000^(2i/d_model) の計算（対数空間で計算して安定性確保）
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )  # (d_model/2,)
        
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数次元
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数次元
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model) - バッチ次元追加
        self.register_buffer('pe', pe)  # 学習パラメータではないのでbufferとして登録
        
    def forward(self, x):
        """
        x : Tensor, shape (batch, seq_len, d_model)
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# テスト
pe = PositionalEncoding(d_model=64, dropout=0.0)
dummy = torch.zeros(1, 100, 64)  # 100位置分のPE
pe_values = pe(dummy)[0].numpy()  # (100, 64)

print(f"PE shape: {pe_values.shape}")
print(f"値の範囲: [{pe_values.min():.4f}, {pe_values.max():.4f}]")

In [ ]:
# ============================================================
# 4.4 Positional Encodingの可視化
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) PE全体のヒートマップ
im0 = axes[0, 0].imshow(pe_values[:50, :32], cmap='RdBu_r', aspect='auto')
axes[0, 0].set_title('Positional Encoding ヒートマップ\n(位置0-49, 次元0-31)', fontsize=12)
axes[0, 0].set_xlabel('次元')
axes[0, 0].set_ylabel('位置')
plt.colorbar(im0, ax=axes[0, 0])

# (b) 特定の次元の波形
for dim_idx in [0, 1, 4, 5, 20, 21]:
    label = f"dim {dim_idx} ({'sin' if dim_idx % 2 == 0 else 'cos'})"
    axes[0, 1].plot(pe_values[:100, dim_idx], label=label, alpha=0.8)
axes[0, 1].set_title('各次元のPositional Encoding波形', fontsize=12)
axes[0, 1].set_xlabel('位置')
axes[0, 1].set_ylabel('値')
axes[0, 1].legend(fontsize=8, loc='upper right')
axes[0, 1].grid(True, alpha=0.3)

# (c) 位置間のコサイン類似度
from numpy.linalg import norm

n_pos = 50
cos_sim = np.zeros((n_pos, n_pos))
for i in range(n_pos):
    for j in range(n_pos):
        cos_sim[i, j] = np.dot(pe_values[i], pe_values[j]) / (
            norm(pe_values[i]) * norm(pe_values[j]) + 1e-8
        )

im2 = axes[1, 0].imshow(cos_sim, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1, 0].set_title('位置間のコサイン類似度', fontsize=12)
axes[1, 0].set_xlabel('位置')
axes[1, 0].set_ylabel('位置')
plt.colorbar(im2, ax=axes[1, 0])

# (d) 特定位置のPEベクトル比較
positions_to_show = [0, 5, 10, 25, 49]
for pos in positions_to_show:
    axes[1, 1].plot(pe_values[pos, :32], label=f'位置 {pos}', alpha=0.8)
axes[1, 1].set_title('各位置のPEベクトル（次元0-31）', fontsize=12)
axes[1, 1].set_xlabel('次元')
axes[1, 1].set_ylabel('値')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Positional Encoding の特性', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("観察:")
print("  - 低次元ほど周波数が高い（位置変化に敏感）")
print("  - 高次元ほど周波数が低い（大域的な位置情報）")
print("  - 近い位置ほどコサイン類似度が高い（対角線付近が明るい）")

# ============================================================
# 5. Transformerブロックの実装
# ============================================================

## 5.1 Transformerブロックの構造

Transformerは **エンコーダ** と **デコーダ** で構成されます。各ブロックの構造は：

### エンコーダブロック
```
Input
  → Multi-Head Self-Attention
  → Add & LayerNorm (残差接続)
  → Feed-Forward Network
  → Add & LayerNorm (残差接続)
→ Output
```

### デコーダブロック
```
Input
  → Masked Multi-Head Self-Attention
  → Add & LayerNorm
  → Multi-Head Cross-Attention (エンコーダ出力を参照)
  → Add & LayerNorm
  → Feed-Forward Network
  → Add & LayerNorm
→ Output
```

### Feed-Forward Network (FFN)
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

通常、内部次元 $d_{ff} = 4 \times d_{\text{model}}$ とします。

In [ ]:
# ============================================================
# 5.2 Feed-Forward Network
# ============================================================

class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network"""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

# ============================================================
# 5.3 Layer Normalization
# ============================================================
# PyTorch標準の nn.LayerNorm を使用

# ============================================================
# 5.4 エンコーダブロック
# ============================================================

class EncoderBlock(nn.Module):
    """Transformer エンコーダブロック"""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Self-Attention + 残差接続 + LayerNorm
        attn_out, attn_weights = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_out))
        
        # Feed-Forward + 残差接続 + LayerNorm
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout2(ff_out))
        
        return x, attn_weights

# ============================================================
# 5.5 デコーダブロック
# ============================================================

class DecoderBlock(nn.Module):
    """Transformer デコーダブロック"""
    
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads)
        self.cross_attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # Masked Self-Attention
        attn_out, self_attn_weights = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(attn_out))
        
        # Cross-Attention (デコーダのQに対して、エンコーダのK, Vを参照)
        attn_out, cross_attn_weights = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout2(attn_out))
        
        # Feed-Forward
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout3(ff_out))
        
        return x, self_attn_weights, cross_attn_weights

# テスト
d_model = 32
n_heads = 4
d_ff = 128
batch_size = 2
src_len = 6
tgt_len = 4

encoder_block = EncoderBlock(d_model, n_heads, d_ff)
decoder_block = DecoderBlock(d_model, n_heads, d_ff)

src = torch.randn(batch_size, src_len, d_model)
tgt = torch.randn(batch_size, tgt_len, d_model)

enc_out, enc_attn = encoder_block(src)
dec_out, dec_self_attn, dec_cross_attn = decoder_block(tgt, enc_out)

print("=== Transformer ブロック テスト ===")
print(f"エンコーダ入力:  {src.shape}")
print(f"エンコーダ出力:  {enc_out.shape}")
print(f"デコーダ入力:   {tgt.shape}")
print(f"デコーダ出力:   {dec_out.shape}")
print(f"\nSelf-Attention重み:  {dec_self_attn.shape}")
print(f"Cross-Attention重み: {dec_cross_attn.shape}")
print(f"\nエンコーダブロック パラメータ数: {sum(p.numel() for p in encoder_block.parameters()):,}")
print(f"デコーダブロック パラメータ数: {sum(p.numel() for p in decoder_block.parameters()):,}")

# ============================================================
# 6. 完全なTransformerモデル
# ============================================================

エンコーダブロックとデコーダブロックを積み重ね、埋め込み層とPositional Encodingを加えて完全なTransformerモデルを構築します。

In [ ]:
# ============================================================
# 6.1 完全な Transformer モデル
# ============================================================

class Transformer(nn.Module):
    """Transformer (Encoder-Decoder)"""
    
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64, n_heads=4,
                 n_encoder_layers=2, n_decoder_layers=2, d_ff=256, dropout=0.1, max_len=100):
        super().__init__()
        
        # 埋め込み層
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        
        # Positional Encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # スケーリング（埋め込みの大きさをPEと揃える）
        self.d_model = d_model
        self.scale = math.sqrt(d_model)
        
        # エンコーダ
        self.encoder_layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_encoder_layers)
        ])
        
        # デコーダ
        self.decoder_layers = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_decoder_layers)
        ])
        
        # 出力層
        self.output_linear = nn.Linear(d_model, tgt_vocab_size)
        
    def encode(self, src, src_mask=None):
        x = self.pos_encoding(self.src_embedding(src) * self.scale)
        for layer in self.encoder_layers:
            x, _ = layer(x, src_mask)
        return x
    
    def decode(self, tgt, enc_output, src_mask=None, tgt_mask=None):
        x = self.pos_encoding(self.tgt_embedding(tgt) * self.scale)
        for layer in self.decoder_layers:
            x, _, _ = layer(x, enc_output, src_mask, tgt_mask)
        return x
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encode(src, src_mask)
        dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask)
        logits = self.output_linear(dec_output)
        return logits
    
    @staticmethod
    def generate_causal_mask(size):
        """因果マスクを生成"""
        mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
        return ~mask  # True=通過, False=遮蔽

# テスト
vocab_size = 20
model = Transformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    d_model=64,
    n_heads=4,
    n_encoder_layers=2,
    n_decoder_layers=2,
    d_ff=256,
    dropout=0.1
)

src = torch.randint(0, vocab_size, (2, 8))   # バッチ2, 系列長8
tgt = torch.randint(0, vocab_size, (2, 6))   # バッチ2, 系列長6
tgt_mask = Transformer.generate_causal_mask(6).unsqueeze(0).unsqueeze(0)  # (1, 1, 6, 6)

logits = model(src, tgt, tgt_mask=tgt_mask)

print("=== 完全な Transformer モデル ===")
print(f"ソース入力:    {src.shape}")
print(f"ターゲット入力: {tgt.shape}")
print(f"出力 logits:   {logits.shape}")
print(f"\n総パラメータ数: {sum(p.numel() for p in model.parameters()):,}")

# モデル構造の概要
print("\n--- モデル構造 ---")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}")

# ============================================================
# 7. 動作確認：文字列コピー・逆順タスク
# ============================================================

Transformerの動作を確認するため、2つの簡単なタスクを用いて学習・評価を行います。

### タスク1: コピータスク
入力系列をそのまま出力する。最もシンプルなSeq2Seqタスク。
- 入力: `[3, 5, 7, 2]` → 出力: `[3, 5, 7, 2]`

### タスク2: 逆順タスク
入力系列を逆順にして出力する。位置情報の理解が必要。
- 入力: `[3, 5, 7, 2]` → 出力: `[2, 7, 5, 3]`

### 特殊トークン
- `0`: PAD（パディング）
- `1`: SOS（Start of Sequence）
- `2`: EOS（End of Sequence）

In [ ]:
# ============================================================
# 7.1 データ生成
# ============================================================

PAD_TOKEN = 0
SOS_TOKEN = 1
EOS_TOKEN = 2
VOCAB_START = 3  # 実際のトークンは3から

def generate_data(n_samples, seq_len, vocab_size, task='copy'):
    """
    コピー/逆順タスクのデータを生成
    
    Parameters:
    -----------
    task : 'copy' or 'reverse'
    """
    src_data = []
    tgt_data = []
    
    for _ in range(n_samples):
        # ランダムな系列を生成（PAD, SOS, EOSを避ける）
        seq = np.random.randint(VOCAB_START, vocab_size, size=seq_len)
        
        if task == 'copy':
            target_seq = seq.copy()
        elif task == 'reverse':
            target_seq = seq[::-1].copy()
        
        # ソース: seq + EOS
        src = list(seq) + [EOS_TOKEN]
        # ターゲット: SOS + target_seq + EOS
        tgt = [SOS_TOKEN] + list(target_seq) + [EOS_TOKEN]
        
        src_data.append(src)
        tgt_data.append(tgt)
    
    return torch.tensor(src_data, dtype=torch.long), torch.tensor(tgt_data, dtype=torch.long)

# データ生成テスト
vocab_size = 12  # 3-11の9種類のトークン
seq_len = 6

src_test, tgt_test = generate_data(3, seq_len, vocab_size, task='copy')
print("=== コピータスク サンプル ===")
for i in range(3):
    print(f"  src: {src_test[i].tolist()}  →  tgt: {tgt_test[i].tolist()}")

print()
src_test_r, tgt_test_r = generate_data(3, seq_len, vocab_size, task='reverse')
print("=== 逆順タスク サンプル ===")
for i in range(3):
    print(f"  src: {src_test_r[i].tolist()}  →  tgt: {tgt_test_r[i].tolist()}")

In [ ]:
# ============================================================
# 7.2 学習ループ
# ============================================================

def train_transformer(task='copy', n_epochs=100, seq_len=6, vocab_size=12,
                      d_model=64, n_heads=4, n_layers=2, d_ff=128,
                      lr=0.001, n_train=500, n_test=100, print_every=20):
    """Transformerを学習する"""
    
    print(f"\n{'='*60}")
    print(f"タスク: {task.upper()}")
    print(f"系列長: {seq_len}, 語彙サイズ: {vocab_size}")
    print(f"d_model: {d_model}, heads: {n_heads}, layers: {n_layers}")
    print(f"{'='*60}")
    
    # データ生成
    src_train, tgt_train = generate_data(n_train, seq_len, vocab_size, task)
    src_test, tgt_test = generate_data(n_test, seq_len, vocab_size, task)
    
    # モデル
    model = Transformer(
        src_vocab_size=vocab_size,
        tgt_vocab_size=vocab_size,
        d_model=d_model,
        n_heads=n_heads,
        n_encoder_layers=n_layers,
        n_decoder_layers=n_layers,
        d_ff=d_ff,
        dropout=0.1
    )
    model.train()
    
    optimizer = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)
    
    # 因果マスク
    tgt_seq_len = tgt_train.size(1)
    tgt_mask = Transformer.generate_causal_mask(tgt_seq_len - 1)
    tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)  # (1, 1, tgt_len-1, tgt_len-1)
    
    losses = []
    accuracies = []
    
    for epoch in range(n_epochs):
        # --- 学習 ---
        optimizer.zero_grad()
        
        # デコーダ入力: SOS + target (EOSを除く)
        tgt_input = tgt_train[:, :-1]  # (batch, tgt_len-1)
        tgt_output = tgt_train[:, 1:]  # (batch, tgt_len-1) - 予測対象
        
        logits = model(src_train, tgt_input, tgt_mask=tgt_mask)
        
        # CrossEntropyLossは (batch*seq_len, vocab) と (batch*seq_len) を期待
        loss = criterion(
            logits.reshape(-1, vocab_size),
            tgt_output.reshape(-1)
        )
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        losses.append(loss.item())
        
        # --- 評価 ---
        if (epoch + 1) % print_every == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                tgt_input_test = tgt_test[:, :-1]
                tgt_output_test = tgt_test[:, 1:]
                
                tgt_mask_test = Transformer.generate_causal_mask(tgt_input_test.size(1))
                tgt_mask_test = tgt_mask_test.unsqueeze(0).unsqueeze(0)
                
                logits_test = model(src_test, tgt_input_test, tgt_mask=tgt_mask_test)
                preds = logits_test.argmax(dim=-1)
                
                # トークン単位の正解率
                correct = (preds == tgt_output_test).float().mean().item()
                # 系列全体の正解率
                seq_correct = (preds == tgt_output_test).all(dim=1).float().mean().item()
                
                accuracies.append((epoch + 1, correct, seq_correct))
                print(f"Epoch {epoch+1:4d} | Loss: {loss.item():.4f} | "
                      f"Token Acc: {correct:.3f} | Seq Acc: {seq_correct:.3f}")
            model.train()
    
    return model, losses, accuracies, (src_test, tgt_test)

# タスク1: コピー
model_copy, losses_copy, acc_copy, test_copy = train_transformer(
    task='copy', n_epochs=150, seq_len=6, vocab_size=12,
    d_model=64, n_heads=4, n_layers=2, d_ff=128, lr=0.001, print_every=25
)

In [ ]:
# タスク2: 逆順
model_reverse, losses_reverse, acc_reverse, test_reverse = train_transformer(
    task='reverse', n_epochs=200, seq_len=6, vocab_size=12,
    d_model=64, n_heads=4, n_layers=2, d_ff=128, lr=0.001, print_every=40
)

In [ ]:
# ============================================================
# 7.3 学習曲線の可視化
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (a) 損失曲線
axes[0].plot(losses_copy, label='Copy', alpha=0.7)
axes[0].plot(losses_reverse, label='Reverse', alpha=0.7)
axes[0].set_title('学習曲線（Loss）', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# (b) トークン正解率
epochs_c, tok_c, seq_c = zip(*acc_copy)
epochs_r, tok_r, seq_r = zip(*acc_reverse)
axes[1].plot(epochs_c, tok_c, 'o-', label='Copy (token)', markersize=4)
axes[1].plot(epochs_r, tok_r, 's-', label='Reverse (token)', markersize=4)
axes[1].set_title('トークン正解率', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.05)

# (c) 系列正解率
axes[2].plot(epochs_c, seq_c, 'o-', label='Copy (sequence)', markersize=4)
axes[2].plot(epochs_r, seq_r, 's-', label='Reverse (sequence)', markersize=4)
axes[2].set_title('系列正解率（完全一致）', fontsize=13)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 1.05)

plt.suptitle('Transformerの学習: コピータスク vs 逆順タスク', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n考察:")
print("  - コピータスクは恒等写像のため比較的簡単")
print("  - 逆順タスクは位置の入れ替えが必要で、Positional Encodingの活用が重要")

In [ ]:
# ============================================================
# 7.4 推論例の確認
# ============================================================

def show_predictions(model, src_data, tgt_data, task_name, n_show=5):
    """予測結果を表示"""
    model.eval()
    print(f"\n=== {task_name} 推論結果 ===")
    
    with torch.no_grad():
        tgt_input = tgt_data[:n_show, :-1]
        tgt_output = tgt_data[:n_show, 1:]
        
        tgt_mask = Transformer.generate_causal_mask(tgt_input.size(1))
        tgt_mask = tgt_mask.unsqueeze(0).unsqueeze(0)
        
        logits = model(src_data[:n_show], tgt_input, tgt_mask=tgt_mask)
        preds = logits.argmax(dim=-1)
        
        correct_count = 0
        for i in range(n_show):
            src_tokens = src_data[i].tolist()
            expected = tgt_output[i].tolist()
            predicted = preds[i].tolist()
            match = "OK" if expected == predicted else "NG"
            if expected == predicted:
                correct_count += 1
            
            print(f"  入力:   {src_tokens}")
            print(f"  期待値: {expected}")
            print(f"  予測:   {predicted}  [{match}]")
            print()
        
        print(f"  表示分の正解率: {correct_count}/{n_show}")

show_predictions(model_copy, *test_copy, "コピータスク", n_show=5)
show_predictions(model_reverse, *test_reverse, "逆順タスク", n_show=5)

# ============================================================
# 8. まとめと次のステップ
# ============================================================

## 本ノートブックで学んだこと

### Transformerの構成要素

| 構成要素 | 役割 | キーポイント |
|---------|------|-------------|
| Self-Attention | トークン間の関係を計算 | Q, K, V行列による重み付き集約 |
| Scaled Dot-Product | スコアの安定化 | $\sqrt{d_k}$ でスケーリング |
| Multi-Head Attention | 複数の観点で注目 | ヘッドごとに異なる関係性を学習 |
| Positional Encoding | 位置情報の付与 | 正弦波関数で任意の長さに対応 |
| Feed-Forward Network | 非線形変換 | 各位置に独立に適用 |
| 残差接続 + LayerNorm | 学習の安定化 | 勾配の流れを確保 |

### RNNとの比較

| 特性 | RNN/LSTM | Transformer |
|------|----------|-------------|
| 計算 | 逐次的 | 並列可能 |
| 長距離依存 | 勾配消失の問題 | 直接的に参照可能 |
| 位置情報 | 構造に暗黙的に含まれる | Positional Encodingで明示的に付与 |
| 計算量 | $O(n \cdot d^2)$ | $O(n^2 \cdot d)$ |

### NB126からの進展
| NB126 | NB127（本ノートブック） |
|-------|----------------------|
| Attention概念の導入 | Multi-Head Attentionの完全実装 |
| Scaled Dot-Product Attention | Positional Encodingの追加 |
| Self-Attentionの概念紹介 | 完全なTransformerモデル構築 |
| Transformerへの布石 | 実際のタスクでの動作確認 |

## 次のステップ
- **事前学習モデル**: BERT, GPTなどのTransformerベースモデル
- **効率的なAttention**: Sparse Attention, Linear Attentionなどの発展手法
- **Vision Transformer (ViT)**: 画像認識へのTransformerの応用

# ============================================================
# 9. 自己評価クイズ
# ============================================================

## 理解度チェック

---

### Q1: Self-Attentionの計算量が $O(n^2 \cdot d)$ である理由は？

<details>
<summary>回答を表示</summary>

$QK^T$ の計算で、$Q \in \mathbb{R}^{n \times d}$ と $K^T \in \mathbb{R}^{d \times n}$ の行列積が $O(n^2 \cdot d)$ かかるため。これは系列長 $n$ の2乗に比例するので、長い系列では計算コストが高くなります。

</details>

---

### Q2: Scaled Dot-Product Attentionで $\sqrt{d_k}$ で割る理由は？

<details>
<summary>回答を表示</summary>

$d_k$ が大きいとき、$Q$ と $K$ の内積の分散が $d_k$ に比例して大きくなります。その結果、softmaxの入力が極端に大きくなり、出力がほぼone-hotになってしまいます（勾配が消失）。$\sqrt{d_k}$ で割ることで分散を1に正規化し、softmaxが適切に機能するようにします。

</details>

---

### Q3: Multi-Head Attentionで $d_k = d_{\text{model}} / h$ とする理由は？

<details>
<summary>回答を表示</summary>

計算コストを単一ヘッドのAttentionと同等に保つためです。$h$ 個のヘッドそれぞれが $d_k = d_{\text{model}}/h$ 次元で計算するので、全ヘッドの計算量は $h \times O(n^2 \cdot d_k) = O(n^2 \cdot d_{\text{model}})$ となり、単一の $d_{\text{model}}$ 次元Attentionと同じです。

</details>

---

### Q4: Positional Encodingで正弦波を使う利点は？（学習可能な埋め込みとの比較）

<details>
<summary>回答を表示</summary>

1. **学習不要**: パラメータを増やさずに位置情報を付与できる
2. **任意の長さに対応**: 学習時に見なかった長さの系列にも対応可能（外挿性）
3. **相対位置の表現**: $PE_{pos+k}$ は $PE_{pos}$ の線形変換で表現でき、相対位置の学習が容易
4. **有界性**: 値が $[-1, 1]$ の範囲に収まる

ただし実践的には、BERT等の多くのモデルでは学習可能な位置埋め込みも広く使われています。

</details>

---

### Q5: Transformerのデコーダで因果マスクが必要な理由は？

<details>
<summary>回答を表示</summary>

Transformerの学習時（Teacher Forcing）では、ターゲット系列全体を一度に入力して並列計算します。しかし推論時は左から右へ逐次的に生成するため、学習時にも「未来のトークンを見ない」制約を課す必要があります。因果マスクにより、各位置は自分自身と過去の位置にのみAttentionを向けることができ、自己回帰的な生成と整合性が保たれます。

</details>

# ============================================================
# 参考文献
# ============================================================

## References

1. **Vaswani, A., et al.** (2017). "Attention Is All You Need." *NeurIPS 2017*. https://arxiv.org/abs/1706.03762
2. **Bahdanau, D., Cho, K., & Bengio, Y.** (2015). "Neural Machine Translation by Jointly Learning to Align and Translate." *ICLR 2015*.
3. **Ba, J. L., Kiros, J. R., & Hinton, G. E.** (2016). "Layer Normalization." *arXiv preprint arXiv:1607.06450*.
4. **The Illustrated Transformer** - Jay Alammar. https://jalammar.github.io/illustrated-transformer/
5. **The Annotated Transformer** - Harvard NLP. https://nlp.seas.harvard.edu/annotated-transformer/

---

**Next**: Transformerの応用（BERT, GPT, Vision Transformerなど）へ進みましょう。